# 📐 UtilScan — Entrena tu detector de útiles escolares

Fusiona 8 datasets públicos de Roboflow Universe (~4.000 fotos, 11 clases), afina **YOLO11s** y lo exporta a `.tflite` para la app.

1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`
2. Consigue tu API key gratis en https://app.roboflow.com/settings/api
3. Ejecuta todas las celdas (~30-40 min en T4)

In [ ]:
!git clone -q https://github.com/AbnerGA7/utilscan.git
%cd utilscan
!pip install -q -r training/requirements.txt
!nvidia-smi -L

## 1. Dataset

`training/merge_datasets.py` descarga los datasets y unifica las clases (*ballpen*, *Ink Pen* → `pen`; *scale*, *penggaris* → `ruler`…). Puedes agregar más datasets editando la lista `DATASETS`.

In [ ]:
import os
os.environ["ROBOFLOW_API_KEY"] = ""  # @param {type:"string"}
!python training/merge_datasets.py
DATA_YAML = "training/datasets/school_supplies/data.yaml"

## 2. Entrenamiento

In [ ]:
from ultralytics import YOLO

EPOCHS = 40  # @param {type:"integer"}
model = YOLO("yolo11s.pt")
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, patience=10, close_mosaic=5,
            degrees=10, flipud=0.1, mixup=0.1, name="school_supplies", exist_ok=True)
metrics = model.val(split="test")
print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")

## 3. Prueba rápida

In [ ]:
import glob
from IPython.display import Image, display
best = YOLO(f"{model.trainer.save_dir}/weights/best.pt")
sample = sorted(glob.glob("training/datasets/school_supplies/test/images/*"))[:3]
for r in best.predict(sample, conf=0.35, save=True):
    display(Image(filename=f"{r.save_dir}/{r.path.split('/')[-1]}", width=480))

## 4. Exportar a TFLite y descargar

In [ ]:
import shutil, pathlib
from google.colab import files

best.export(format="tflite", imgsz=640, half=True)
tflite = next(pathlib.Path(model.trainer.save_dir).rglob("*_float16.tflite"))
shutil.copy(tflite, "school_supplies.tflite")
pathlib.Path("school_supplies_labels.txt").write_text("
".join(best.names[i] for i in range(len(best.names))) + "
")
files.download("school_supplies.tflite")
files.download("school_supplies_labels.txt")

## 5. Usarlo en la app

Copia los dos archivos a `app/src/main/assets/models/` y recompila. UtilScan detecta el modelo automáticamente y lo usa con prioridad (también puedes alternar entre modelos en *Ajustes*).